# 파이썬 업무자동화 ③ 웹 자동화(동적 페이지)

> **실습 파일 사용법**
>
> - **[직접 입력]** 표시가 있는 셀: 바로 위 마크다운 셀의 코드를 보고 직접 입력한 뒤 실행합니다.
> - **[직접 작성]** 표시가 있는 셀: 문제를 읽고 코드를 스스로 작성합니다. 정답은 완성 파일에 있습니다.
> - **[실행만]** 표시가 있는 셀: 코드를 입력하지 않고 그대로 실행합니다.

> **실습 준비**
>
> - 이 노트북 파일의 코드는 크롬 창을 열어 웹사이트에 접속합니다. 컴퓨터에 크롬이 설치되어 있어야 하고, 인터넷에 연결되어 있어야 합니다.
> - 영화 순위와 공연 목록은 조회하는 날마다 달라집니다. 실행 결과에 적힌 내용은 예시입니다.
> - 웹사이트 화면이 바뀌면 칸 이름(id)이나 버튼 글자가 달라져서 코드가 동작하지 않을 수 있습니다. 이때는 F12로 화면 요소를 다시 확인합니다.
> - 교안의 코드는 `r"C:\workspace\python20261103\6일차"`처럼 전체 경로를 씁니다. 노트북에서는 노트북 파일과 같은 폴더에 저장합니다.

# 실습 3. 웹 자동화(동적 페이지)

## 웹 자동화란?

사람이 하던 클릭과 입력을 코드가 대신해, 화면에 뜬 표를 업무 파일로 남깁니다.

| 단계 | 하는 일 | 예시 → 처리 결과 |
|---|---|---|
| 01 브라우저 열기 | 사이트 주소로 접속 | `driver.get(url)` → 크롬 창이 열림 |
| 02 조건 넣고 클릭 | 날짜 입력 → 조회 버튼 | `button.click()` → 결과 표가 나타남 |
| 03 기다렸다 읽기 | 표가 다 뜰 때까지 대기 | `row.text` → `"1 극장판 치이카와 …"` |
| 04 파일로 저장 | CSV·엑셀로 남기기 | `writer.writerow(행)` → 박스오피스.csv |

## 도구: selenium · csv · datetime

| 모듈 | 역할 | 자주 쓰는 기능 |
|---|---|---|
| selenium | 브라우저를 코드로 조작 | `webdriver.Chrome()`, `driver.get()`, `.find_element()`, `.click()` |
| WebDriverWait | 요소가 뜰 때까지 기다리기 | `WebDriverWait(driver, 10).until(EC.presence_of_…)` |
| csv | CSV 파일로 저장 | `csv.writer()`, `.writerow()` |
| datetime | 오늘·어제 날짜 구하기 | `date.today()`, `timedelta(days=1)`, `.strftime("%Y-%m-%d")` |

- selenium만 설치 필요 → 터미널에서 `pip install selenium`
- 크롬 드라이버는 따로 받지 않음 → 크롬만 있으면 selenium이 알아서 준비합니다.

주피터 노트북에서는 아래 셀로 설치할 수 있습니다. 이미 설치되어 있으면 건너뜁니다.

In [ ]:
# [실행만] 이 셀은 입력하지 않고 그대로 실행합니다.
%pip install selenium

## 기본 사용법 ① 브라우저 열고 접속하기

**목표** : 크롬을 코드로 열고 KOBIS 일별 박스오피스로 이동하기

| 기능 | 설명 |
|---|---|
| `webdriver.Chrome()` | 크롬 창을 코드로 열기 |
| `driver.get(url)` | 주소로 이동하기 |
| `driver.quit()` | 다 쓴 창 닫기 |

**주의** : 크롬이 설치돼 있어야 합니다. 첫 실행은 준비하느라 몇 초 더 걸립니다.

교안 코드는 마지막 줄에서 `driver.quit()`로 창을 닫습니다. 노트북에서는 기본 사용법 ②~④에서 같은 창을 계속 쓰기 때문에, `driver.quit()`은 기본 사용법 ④를 마친 뒤 따로 실행합니다.

**[직접 입력]** 아래 코드를 다음 셀에 입력하고 실행합니다.

```python
from selenium import webdriver

url = ("https://www.kobis.or.kr/kobis/business"
       "/stat/boxs/findDailyBoxOfficeList.do")

driver = webdriver.Chrome()   # 크롬 창 열기
driver.get(url)               # 주소로 이동
print("접속 완료")
```

In [ ]:
# [직접 입력] 위 코드를 입력하세요.

실행 결과 : 크롬 창이 열리고 KOBIS 일별 박스오피스 화면이 나타납니다.
```
접속 완료
```

## 기본 사용법 ② 화면 요소 찾기

**목표** : F12로 날짜 칸·조회 버튼의 이름을 찾아 코드로 가리키기

| 기능 | 설명 |
|---|---|
| F12 → 우클릭 → 검사 | 화면 요소의 HTML 코드 보기 |
| `By.ID` | `id="..."` 이름으로 찾기 |
| `By.XPATH` | 화면 글자로 찾기 |

**참고** : 칸 위에서 우클릭 → [검사]를 고르면 그 칸의 코드가 바로 보입니다.

**[직접 입력]** 아래 코드를 다음 셀에 입력하고 실행합니다.

```python
from selenium.webdriver.common.by import By

# 시작 날짜 입력칸 (id로 찾기)
box = driver.find_element(By.ID, "sSearchFrom")

# 조회 버튼 (글자로 찾기)
button = driver.find_element(By.XPATH,
             "//button[text()='조회']")

print(box.tag_name, button.text)
```

In [ ]:
# [직접 입력] 위 코드를 입력하세요.

실행 결과
```
input 조회
```

## 기본 사용법 ③ 날짜 넣고 조회 누르기

**목표** : 시작·끝 날짜 칸에 같은 날짜를 넣고 조회 버튼 클릭하기

| 기능 | 설명 |
|---|---|
| `execute_script(...)` | 입력칸에 값을 직접 넣기 |
| `.click()` | 버튼 누르기 |
| `sSearchFrom` · `sSearchTo` | 기간 시작·끝 칸 (같은 날 → 하루치) |

**주의** : 달력 칸은 키보드 입력이 막혀 있을 수 있어 `send_keys()` 대신 값을 직접 넣습니다.

**[직접 입력]** 아래 코드를 다음 셀에 입력하고 실행합니다.

```python
day = "2026-11-07"                 # 조회할 날짜

for name in ["sSearchFrom", "sSearchTo"]:
    box = driver.find_element(By.ID, name)
    driver.execute_script(
        "arguments[0].value = arguments[1];",
        box, day)                    # 칸에 값 넣기

button.click()                     # 조회 클릭
print("조회:", day)
```

In [ ]:
# [직접 입력] 위 코드를 입력하세요.

실행 결과 : 크롬 창의 표가 2026-11-07 순위로 바뀝니다.
```
조회: 2026-11-07
```

## 기본 사용법 ④ 표가 뜰 때까지 기다렸다 읽기

**목표** : 결과 표가 나타나면 1~3위의 순위와 영화명 출력하기

| 기능 | 설명 |
|---|---|
| `wait.until(...)` | 표 첫 줄(tr_0)이 뜰 때까지 기다리기 |
| `find_elements` | 여러 개를 리스트로 찾기 |
| `cells[번호].text` | 칸에 보이는 글자 꺼내기 |

**주의** : 기다리지 않고 바로 찾으면 표가 아직 없어 NoSuchElementException이 납니다.

**[직접 입력]** 아래 코드를 다음 셀에 입력하고 실행합니다.

```python
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

wait = WebDriverWait(driver, 10)         # 최대 10초
wait.until(EC.presence_of_element_located((By.ID, "tr_0")))

rows = driver.find_elements(By.CSS_SELECTOR, "tr[id^='tr_']")
for row in rows[:3]:                     # 1~3위
    cells = row.find_elements(By.TAG_NAME, "td")
    print(cells[0].text, cells[1].text)  # 순위, 영화명
```

In [ ]:
# [직접 입력] 위 코드를 입력하세요.

실행 결과 (예시)
```
1 극장판 치이카와: 인어 섬의 비밀
2 암살자(들)
3 타짜: 벨제붑의 노래
```

**창 닫기** : 기본 사용법을 마쳤으면 크롬 창을 닫습니다.

**[직접 입력]** 아래 코드를 다음 셀에 입력하고 실행합니다.

```python
driver.quit()                 # 창 닫기
```

In [ ]:
# [직접 입력] 위 코드를 입력하세요.

## 예제 ① 일별 박스오피스 수집기

**자동화 전: 매일 아침 직접 정리**
1. KOBIS 사이트 접속
2. 일별 박스오피스 메뉴 찾기
3. 어제 날짜 고르고 조회
4. 1~10위를 엑셀에 옮겨 적기
5. 관객 수 순으로 다시 정렬

⋯ 매일 아침 반복

**자동화 후: 실행 한 번으로 CSV 완성 — 영화_박스오피스_20261107.csv**
```
순위,영화명,관객수,누적관객수,매출액
1,극장판 치이카와,146248,…
2,암살자(들),56420,…
3,타짜: 벨제붑의 노래,36249,…
  ⋯ 10편, 관객 수 많은 순
```

**자동화 규칙** : 어제 날짜로 조회 → 표가 뜰 때까지 기다리기 → 1~10위 값 꺼내기 → 관객 수 순으로 정렬해 CSV 저장

### 예제 ① 사용할 사이트: KOBIS 일별 박스오피스

**KOBIS** : 날짜만 고르면 전국 영화 흥행 순위를 표로 보여 주는 공식 사이트

**화면 읽는 법**
```python
url = ("https://www.kobis.or.kr/kobis/business"
       "/stat/boxs/findDailyBoxOfficeList.do")
# 코드로 가리킬 화면 요소
(By.ID, "sSearchFrom")                 # 시작 날짜 칸
(By.ID, "sSearchTo")                   # 끝 날짜 칸
(By.XPATH, "//button[text()='조회']")  # 조회 버튼
(By.CSS_SELECTOR, "tr[id^='tr_']")     # 표의 한 줄
```

| 항목 | 설명 |
|---|---|
| 조회 기간 | 시작·끝을 같은 날로 → 하루치 |
| 결과 표 | 1위부터 한 줄씩 tr_0, tr_1 … |
| 집계 기준 | 어제까지 집계, 오늘 값은 아직 없음 |

**표 한 줄의 칸 번호 (일부)**

| 칸 | 내용 |
|---|---|
| td[0] | 순위 |
| td[1] | 영화명 |
| td[3] | 매출액 |
| td[8] | 관객수 |
| td[11] | 누적관객수 |

**참고** : 칸 이름과 번호는 F12로 확인합니다. (kobis.or.kr → 박스오피스 → 일별)

### 예제 ① 풀이: 단계별로 생각하기

- **입력** : 어제 날짜·KOBIS 주소
- **처리** : 날짜 넣고 조회 → 표를 기다려 1~10위 꺼내기
- **출력** : 관객 수 순 CSV 파일

| 단계 | 하는 일 | 코드 |
|---|---|---|
| 1. 어제 날짜 만들기 | 오늘에서 하루를 빼고 "2026-11-07" 모양으로 | `yesterday = date.today() - timedelta(days=1)`<br>`day = yesterday.strftime("%Y-%m-%d")` |
| 2. 날짜 넣고 조회 누르기 | 기본 사용법 ③과 같은 방법 | `driver.execute_script(..., box, day)`<br>`button.click()` |
| 3. 1~10위 값 꺼내기 | 관객 수는 쉼표를 빼야 숫자로 정렬됨 | `cells = row.find_elements(By.TAG_NAME, "td")`<br>`audience = int(cells[8].text.replace(",", ""))` |
| 4. 관객 수 순으로 정렬해 저장 | 큰 수부터 정렬하려면 `reverse=True` | `movies.sort(key=lambda m: m[2], reverse=True)`<br>`writer.writerows(movies)` |

### 예제 ① 풀이: 완성 코드

**[직접 입력]** 아래 코드를 다음 셀에 입력하고 실행합니다.

```python
import csv, time
from datetime import date, timedelta
from selenium import webdriver
from selenium.webdriver.common.by import By

# 교안: folder = r"C:\workspace\python20261103\6일차"
# 노트북에서는 노트북 파일과 같은 폴더에 저장합니다.
yesterday = date.today() - timedelta(days=1)
day = yesterday.strftime("%Y-%m-%d")  # 어제
name = f"영화_박스오피스_{yesterday:%Y%m%d}.csv"

url = ("https://www.kobis.or.kr/kobis/business"
       "/stat/boxs/findDailyBoxOfficeList.do")

def num(cell):                 # "1,234" → 1234
    return int(cell.text.replace(",", ""))

driver = webdriver.Chrome()
driver.get(url)

for name_id in ["sSearchFrom", "sSearchTo"]:
    box = driver.find_element(By.ID, name_id)
    driver.execute_script(
        "arguments[0].value=arguments[1]", box, day)

xpath = "//button[text()='조회']"
driver.find_element(By.XPATH, xpath).click()
time.sleep(2)                  # 새 표로 바뀔 시간

rows = driver.find_elements(By.CSS_SELECTOR,
                            "tr[id^='tr_']")
movies = []
for row in rows[:10]:              # 1~10위
    td = row.find_elements(By.TAG_NAME, "td")
    movies.append([td[0].text, td[1].text,
        num(td[8]), num(td[11]), num(td[3])])

movies.sort(key=lambda m: m[2], reverse=True)

with open(name, "w", newline="",
          encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(["순위", "영화명", "관객수",
                     "누적관객수", "매출액"])
    writer.writerows(movies)

driver.quit()
print("저장 완료:", len(movies), "편")
```

In [ ]:
# [직접 입력] 위 코드를 입력하세요.

실행 결과
```
저장 완료: 10 편
```

노트북 파일과 같은 폴더에 `영화_박스오피스_날짜.csv`가 생깁니다. 엑셀로 열어 관객 수가 많은 순인지 확인합니다.

## 실습 미션: 일주일 순위를 CSV 하나로 저장하기

**미션** : 예제 ① 코드를 고쳐서, 최근 7일치 1~10위를 주간_박스오피스.csv에 저장해 보세요.

**이렇게 저장되면 성공: 주간_박스오피스.csv**
```
날짜,순위,영화명,관객수
2026-11-01,1,극장판 치이카와,…
2026-11-01,2,암살자(들),…
  ⋯
2026-11-07,10,…
⋯ 7일 × 10편 = 70줄
```

**힌트** : 날짜를 바꿔 가며 조회를 7번 반복하기

1. 7일 전부터 어제까지 날짜 만들기
```python
for i in range(7, [ ? ], -1):
    d = date.today() - timedelta(days=i)
```
2. 날짜마다 조회하고 1~10위 꺼내기
```python
    day = d.strftime("%Y-%m-%d")
    movies.append([day, td[0].text,
                   td[1].text, [ ? ]])
```
3. 저장은 반복이 끝난 뒤 한 번만
```python
writer.writerows(movies)
```

In [ ]:
# [직접 작성] 여기에 코드를 작성하세요.

## 예제 ① AI 어시스턴트로 코드 만들기

**프롬프트 뼈대**

| 칸 | 적는 내용 |
|---|---|
| 목적 | 무엇을 하는 코드인지 |
| 입력 | 어떤 데이터를 받아서 |
| 처리 | 어떤 기준으로 처리할지 |
| 출력 | 결과를 어떻게 보여줄지 |
| 조건 | 사용할 도구, 지킬 제약 |

**예시 프롬프트**
```
[목적] KOBIS 일별 박스오피스에서 어제 영화 순위를 모아 CSV로 저장하는 파이썬 코드를 만들어 줘.
[입력] kobis.or.kr 일별 박스오피스 화면, 날짜 칸 id는 sSearchFrom·sSearchTo, 조회 버튼
[처리] 날짜 칸에 어제 날짜를 넣고 조회 → 표가 뜨면 1~10위의 순위·영화명·관객 수·누적 관객 수·매출액 꺼내기
[출력] 관객 수가 많은 순으로 정렬해 영화_박스오피스_날짜.csv로 저장하고, 초보자용 주석을 달아 줘.
[조건] selenium과 csv만 사용. 표를 기다리는 코드를 넣고, 저장 폴더는 내가 붙여 넣을 수 있게 맨 위에 둬.
```

AI가 준 코드를 아래 셀에 붙여 넣고 실행해 봅니다.

In [ ]:
# AI가 준 코드를 붙여 넣고 실행합니다.

## 연습문제에 사용할 사이트: KOPIS 공연예술통합전산망

**KOPIS** : 장르와 기간을 고르면 전국 공연 목록을 표로 보여 주는 공식 사이트

**예제 ①과 비교**
```python
url = ("https://www.kopis.or.kr/por/db/pblprfr"
       "/pblprfr.do?menuId=MNU_00020")
# 사람이 하던 일 → 코드
# ① 장르 '뮤지컬' 체크 → .click()
# ② 공연기간 1일~말일 → execute_script(...)
# ③ 검색 버튼 누르기 → .click()
# ④ 결과 표의 줄 읽기 → find_elements(...)
```

| 구분 | 내용 |
|---|---|
| 같은 점 | 칸에 값 넣기 → 검색 클릭 → 표 읽기 |
| 다른 점 | 조건이 장르·기간 두 가지 |
| 이번 달 1일 | `date.today().replace(day=1)` |

**결과 표 한 줄에 담긴 정보** : 공연명 | 공연기간 | 공연장 | 지역 | 장르 → 한 줄이 공연 하나

**참고** : 장르 체크칸과 기간 칸의 이름은 F12로 직접 찾아 넣습니다. (kopis.or.kr)

## 연습문제: AI 어시스턴트로 이번 달 뮤지컬 목록 만들기

**문제** : AI 어시스턴트로 코드를 만들어 이번 달 뮤지컬 공연 목록을 CSV로 저장해 보세요.

**사람이 하던 일 (화면)**
```
KOPIS 공연 검색
장르  ☑ 뮤지컬
기간  11-01 ~ 11-30
[검색] 클릭
→ 공연명 | 기간 | 공연장 | 지역 …  ⋯ 여러 줄
```

**실행 후: 이번달_뮤지컬_공연목록.csv**
```
공연명,공연기간,공연장,지역
공연 A,11.01~12.28,…,서울
공연 B,10.15~11.30,…,부산
공연 C,11.07~11.09,…,대구
  ⋯
```
한 줄 = 공연 하나

**조건**

| 항목 | 내용 |
|---|---|
| 주소 | kopis.or.kr 공연 검색 화면 |
| 장르 | 뮤지컬 |
| 기간 | 이번 달 1일 ~ 말일 |
| 저장 | 공연명·기간·공연장·지역 |

※ 결과가 여러 쪽이면 첫 쪽만 저장해도 됩니다

**프롬프트 뼈대에 채워 넣기** : [목적] [입력] [처리] [출력] [조건] — 조건 표의 내용을 칸마다 나눠 적기

In [ ]:
# AI가 준 코드를 붙여 넣거나, 직접 작성합니다.

# [직접 작성] 여기에 코드를 작성하세요.